In [18]:
# 06.09.2026 - 20260906a.ipynb: FUNDAMENT PRZELICZENIA W KONWENCJI ARTYKULU.
#
# JEDEN watek: wprowadzic wspolny modul statystyki, udowodnic, ze odtwarza on
# OBIE konwencje co do bitu, i policzyc nim krzywe odniesienia, na ktorych
# oprze sie caly reszta przeliczenia (notebooki b-g).
#
# DLACZEGO TO POWSTAJE. 05.09 (20260905c.ipynb, dziennik 20260905.txt pkt
# 13a-13g) ustalilismy, ze kod zespolu - a wiec i artykul - uzywa innej
# konwencji binowania niz nasza: okno zaczyna sie o jeden kosz przed t0, krata
# jest zakotwiczona w pierwszym dostepnym rekordzie zamiast w t0 (co daje
# efektywne dt = 14,79 dnia zamiast 15), mediany licza sie po filtrze, filtr
# jest skrocony, a remisy zostaja w mianowniku. Po przejsciu na te konwencje
# odtwarzamy opublikowane glebokosci wszystkich pieciu minimow sekcji 4 z
# dokladnoscia 0,24-0,77 dekady, zamiast 0,24-3,25 jak wczesniej.
#
# DECYZJE PRZYJETE PRZED PISANIEM TEGO KODU (Maciek, 05.09):
#  1. KONWENCJA idzie wszedzie. WERSJA PLIKU CR z archiwum CREDO - tylko tam,
#     gdzie zestawiamy nasze liczby z opublikowanymi (rysunek slajdu 2,
#     slajd dodatkowy o sekcji 3, outlook). Pozostale rachunki zostaja na
#     naszym pobraniu z NMDB, bo plik z archiwum istnieje wylacznie dla
#     Moskwy, Oulu i Augera - dla pozostalych 19 stacji nie ma czego podmienic
#     - i konczy sie 2019-12-11, co skrocilo by krate t0 o piec lat.
#     Uzasadnienie merytoryczne: caly lancuch surogatowy porownuje prawdziwy
#     katalog z surogatami przez TEN SAM plik CR, wiec wersja pliku dziala tam
#     po obu stronach i sie skraca.
#  2. Kalibracja na 500 surogatach (20260902a) NIE jest przeliczana - zostaje
#     z adnotacja, ze policzona w starej konwencji.
#  3. Kod idzie do wspolnego modulu ../statystyka.py, a nie do przelacznika w
#     kazdym notebooku. Do 05.09 ta sama funkcja byla skopiowana w dziewieciu
#     notebookach.
#
# CZEGO TEN NOTEBOOK NIE ROBI. Nie interpretuje niczego - liczy krzywe i
# sprawdza, ze liczy je poprawnie. Ery, istotnosc, surogaty i wszystkie wnioski
# sa w notebookach b-g.
#
# WEJSCIA:
#   ../data/csv_data_stations_extended/{stacja}_extended_6h.csv  (20 stacji)
#   ../data/csv_data_stations_full6h/{stacja}_full_6h.csv        (uzupelnienie)
#   ../data/mosc_data.csv                                        (NMDB, nasze)
#   ../zrodla/kod/dane/mosc_data2.csv                            (archiwum CREDO)
#   ../data/usgs_data/usgs_m4_1965_2025.csv                      (USGS M >= 4,0)
#   ../zrodla/kod/dane/{eq_data.csv, pdf_Mosc_2013-11-14 07-00-00.csv}
#   ../results/finetune_{stacja}_P3350_d5_dt15_fullhistory.csv   (stara konwencja)
# WYJSCIA:
#   ../results/credo_{stacja}_P3350_d5_dt15_fullhistory.csv      (20 plikow)
#   ../results/credo_mosc_archiwum_P3350_d5_dt15_fullhistory.csv
#   ../results/credo_walidacja.csv, credo_kotwice.csv
import os
import sys
import time

import numpy as np
import pandas as pd

sys.path.insert(0, "..")
import importlib

import statystyka as st
importlib.reload(st)   # statystyka.py jest w trakcie rozwoju; bez reload jadro
                       # trzyma wersje z pierwszego importu i zmiany na dysku sa
                       # niewidoczne az do restartu kernela
from mc_parallel import run_jobs_parallel

RESULTS_DIR = "../results"
ARCH_DIR = "../źródła/kod/dane/"
EXTENDED_DIR = "../data/csv_data_stations_extended"
FULL6H_DIR = "../data/csv_data_stations_full6h"
MOSC_NASZ_PATH = "../data/mosc_data.csv"
MOSC_ARCH_PATH = ARCH_DIR + "mosc_data2.csv"
OULU_PATH = "../data/oulu_5min_data.csv"
USGS_PATH = "../data/usgs_data/usgs_m4_1965_2025.csv"
EQ_ARCH_PATH = ARCH_DIR + "eq_data.csv"
TABELA_ZESPOLU = ARCH_DIR + "pdf_Mosc_2013-11-14 07-00-00.csv"

P_DAYS, D_DAYS, M_THRESHOLD, DT_DAYS = 3350, 5, 4.0, 15
P3_DAYS = 1675
T0_SEKCJA3 = pd.Timestamp("2013-11-14 07:00:00")

OKNO_STARA = st.okno(P_DAYS, D_DAYS, DT_DAYS, st.STARA)
OKNO_CREDO = st.okno(P_DAYS, D_DAYS, DT_DAYS, st.CREDO)
OKNO3_CREDO = st.okno(P3_DAYS, D_DAYS, DT_DAYS, st.CREDO)


def load_oulu():
    # Przepisane znak w znak z 20260826d. errors="coerce" + dropna omijaja znany
    # uszkodzony wiersz pliku 5-minutowego (CLAUDE.md, linia 839815).
    df = pd.read_csv(OULU_PATH)
    df["datetime"] = pd.to_datetime(df["datetime"], errors="coerce")
    df = df.dropna(subset=["datetime"])
    return df.set_index("datetime").sort_index()["value"].resample("6h").mean()


def load_station(station):
    # UWAGA: mosc i oulu NIE leza w katalogach stacji - maja wlasne pliki i
    # wlasne loadery, dokladnie tak jak w 20260826d, ktory policzyl krzywe
    # finetune_*. Bez tego rozgalezienia zadanie dla tych dwoch stacji probuje
    # czytac nieistniejacy csv_data_stations_full6h/{oulu,mosc}_full_6h.csv.
    if station.lower() == "mosc":
        return load_cr(MOSC_NASZ_PATH)
    if station.lower() == "oulu":
        return load_oulu()
    ext = os.path.join(EXTENDED_DIR, f"{station.lower()}_extended_6h.csv")
    full = os.path.join(FULL6H_DIR, f"{station.lower()}_full_6h.csv")
    df = pd.read_csv(ext if os.path.exists(ext) else full)
    df["datetime"] = pd.to_datetime(df["datetime"])
    s = df.set_index("datetime").sort_index()["value"]
    return s[~s.index.duplicated(keep="first")]


def load_cr(path):
    df = pd.read_csv(path)
    df["datetime"] = pd.to_datetime(df["datetime"])
    s = df.set_index("datetime").sort_index()["value"]
    return s[~s.index.duplicated(keep="first")]


def load_eq(path, utc=False, min_mag=M_THRESHOLD):
    df = pd.read_csv(path, usecols=["time", "mag"])
    df["time"] = (pd.to_datetime(df["time"], utc=True).dt.tz_localize(None) if utc
                  else pd.to_datetime(df["time"]))
    return df[df["mag"] >= min_mag].set_index("time")["mag"].sort_index()


# Stacje: te, dla ktorych 26.08 policzono krzywe pelnej historii. Lista bierze
# sie z plikow, a nie z recznego spisu, zeby nie rozjechala sie po cichu.
STACJE = sorted(f.split("_")[1] for f in os.listdir(RESULTS_DIR)
                if f.startswith("finetune_") and f.endswith("P3350_d5_dt15_fullhistory.csv"))

eq = load_eq(USGS_PATH, utc=True)
eq_prep = st.przygotuj_katalog(eq)

# SIATKA t0 JEST PER STACJA, NIE WSPOLNA. 20260826d skanowal kazda stacje po
# jej WLASNYM zakresie dat (od poczatku jej danych do momentu, w ktorym cale
# okno P + dt jeszcze sie miesci), wiec pliki finetune_* maja 15 roznych
# dlugosci: od 11 596 punktow (psnm, dane od 2007) do 74 322 (mosc, jung, newk,
# sopo, thul). Lacznie ~941 tys. kandydatow t0 - to jest ta liczba, ktora
# prezentacja cytuje na slajdzie 2.
#
# Krzywe credo_* licza sie na DOKLADNIE tych samych siatkach, co odpowiadajace
# im pliki finetune_*. Dwa powody: (1) stara i nowa krzywa daja sie wtedy odjac
# punkt po punkcie, (2) liczenie stacji poza zakresem jej danych produkuje
# smieci - okno z kilkudziesiecioma waznymi koszami potrafi dac -log10(PPDF)
# rzedu 12-16, glebiej niz cokolwiek prawdziwego, i find_peaks by to lykal.
def siatka_stacji(stacja):
    df = pd.read_csv(f"{RESULTS_DIR}/finetune_{stacja}_P3350_d5_dt15_fullhistory.csv",
                     usecols=["t0"], parse_dates=["t0"]).sort_values("t0")
    return pd.DatetimeIndex(df["t0"].reset_index(drop=True))


SIATKA = siatka_stacji("mosc")          # siatka Moskwy - uzywana w komorkach 1, 3 i 5
SIATKA_NS = SIATKA.values.astype("datetime64[ns]").astype(np.int64)
_ft_mosc = pd.read_csv(f"{RESULTS_DIR}/finetune_mosc_P3350_d5_dt15_fullhistory.csv",
                       parse_dates=["t0"]).sort_values("t0").reset_index(drop=True)

# PROG HIGIENY SKANU. Kotwiczenie na danych przy skanie po calej historii cicho
# przenosi okno przez dziury w szeregu CR - i wynik wyglada na policzony. Punkty,
# dla ktorych efektywne dt odbiega od nominalnego o wiecej niz szerokosc kosza,
# dostaja NaN. To NIE jest zmiana statystyki, tylko odmowa liczenia okna, ktore
# nie jest tym oknem, o ktore pytamy. Uzasadnienie: docstring st.krzywa.
MAX_JITTER_D = D_DAYS

_brak = [x for x in STACJE if x.lower() not in ("mosc", "oulu")
         and not os.path.exists(os.path.join(EXTENDED_DIR, f"{x.lower()}_extended_6h.csv"))
         and not os.path.exists(os.path.join(FULL6H_DIR, f"{x.lower()}_full_6h.csv"))]
assert not _brak, f"stacje bez pliku CR: {_brak}"
print(f"Stacje: {len(STACJE)} -> {', '.join(STACJE)}")
print(f"Katalog EQ: {len(eq)} zdarzen M >= {M_THRESHOLD}, "
      f"{eq.index.min().date()} .. {eq.index.max().date()}")
_dl = {x: len(siatka_stacji(x)) for x in STACJE}
print(f"Siatki t0 per stacja: {len(set(_dl.values()))} roznych dlugosci, "
      f"lacznie {sum(_dl.values())} kandydatow; "
      f"min {min(_dl.values())} ({min(_dl, key=_dl.get)}), "
      f"max {max(_dl.values())} ({max(_dl, key=_dl.get)})")
print(f"Prog higieny skanu: |dt_efektywne - {DT_DAYS} d| <= {MAX_JITTER_D} d")
print(f"Siatka Moskwy: {len(SIATKA)} punktow, krok "
      f"{(SIATKA[1] - SIATKA[0]) / pd.Timedelta(hours=1):.0f} h, "
      f"{SIATKA[0]} .. {SIATKA[-1]}")
print(f"Okno STARA: {OKNO_STARA.n_koszy} koszy -> {OKNO_STARA.n_koszy - 1} par")
print(f"Okno CREDO: {OKNO_CREDO.n_koszy} koszy -> {OKNO_CREDO.n_koszy - 1} par")

Stacje: 20 -> aatb, apty, athn, fsmt, hrms, invk, jung, jung1, lmks, mosc, mxco, nain, newk, oulu, psnm, pwnk, sopb, sopo, tera, thul
Katalog EQ: 507919 zdarzen M >= 4.0, 1965-01-01 .. 2025-01-31
Siatki t0 per stacja: 15 roznych dlugosci, lacznie 940792 kandydatow; min 11596 (psnm), max 74322 (jung)
Prog higieny skanu: |dt_efektywne - 15 d| <= 5 d
Siatka Moskwy: 74322 punktow, krok 6 h, 1965-01-01 12:00:00 .. 2015-11-15 18:00:00
Okno STARA: 670 koszy -> 669 par
Okno CREDO: 671 koszy -> 670 par


In [19]:
# KOMORKA 1 - WALIDACJA A: czy modul odtwarza STARA konwencje, czyli to, czym
# policzono wszystkie pliki finetune_*.csv (26.08, implementacja referencyjna
# cosmoseismic_stat) i cala prace surogatowa z 02-04.09 (implementacja szybka).
#
# Dwa osobne testy, bo w starej konwencji te dwie implementacje NIE sa wymienne
# co do bitu (filtr rownania (3) odrzuca kosze bitowo rowne medianie - dziennik
# 20260905.txt pkt 10f):
#   A1  referencyjna z modulu kontra zapisany plik    -> ma byc DOKLADNIE to samo
#   A2  szybka z modulu kontra ten sam plik           -> ma byc blisko, z
#       rozbieznoscia tego samego rzedu, co mierzyly notebooki z 02-04.09
_t = time.time()
mosc_nasz = load_cr(MOSC_NASZ_PATH)
cr_prep_mosc = st.przygotuj_szereg(mosc_nasz)

rng = np.random.default_rng(20260906)
_probka = np.sort(rng.choice(len(SIATKA), size=200, replace=False))

print("A1. Referencyjna z modulu (STARA) kontra finetune_mosc_...fullhistory.csv")
_ref_rows = []
for i in _probka:
    r = st.stat_ref(mosc_nasz, eq, SIATKA[i], P_DAYS, D_DAYS, M_THRESHOLD, DT_DAYS,
                    konw=st.STARA)
    _ref_rows.append(dict(i=i, Np_mod=r["Np"], Nm_mod=r["Nm"], PPDF_mod=r["PPDF"]))
_a1 = pd.DataFrame(_ref_rows).merge(
    _ft_mosc.reset_index().rename(columns={"index": "i"})[["i", "Np", "Nm", "PPDF"]], on="i")
_a1["d_log10"] = np.abs(-np.log10(_a1["PPDF_mod"]) + np.log10(_a1["PPDF"]))
print(f"   punktow: {len(_a1)}; z innym Np: {int((_a1['Np_mod'] != _a1['Np']).sum())}; "
      f"z innym Nm: {int((_a1['Nm_mod'] != _a1['Nm']).sum())}; "
      f"max |d log10 PPDF|: {_a1['d_log10'].max():.3e}")
assert (_a1["Np_mod"] == _a1["Np"]).all() and (_a1["Nm_mod"] == _a1["Nm"]).all(), \
    "modul w konwencji STARA nie odtwarza cosmoseismic_stat - nic dalej nie ma sensu"

print("\nA2. Szybka z modulu (STARA) kontra ten sam plik, co dziesiaty punkt siatki")
_co10 = np.arange(0, len(SIATKA), 10)
y_szybka, Np_sz, Nm_sz, N_sz, _ = st.krzywa(cr_prep_mosc, eq_prep, SIATKA_NS[_co10], OKNO_STARA)
_y_ref = -np.log10(_ft_mosc["PPDF"].to_numpy()[_co10])
_ok = ~np.isnan(y_szybka) & ~np.isnan(_y_ref)
_d = np.abs(y_szybka - _y_ref)
print(f"   punktow: {len(_co10)}; korelacja {np.corrcoef(y_szybka[_ok], _y_ref[_ok])[0, 1]:.6f}; "
      f"|roznica| > 0,2 w {int(np.nansum(_d > 0.2))} punktach; max {np.nanmax(_d):.3f}")
print("   (rozbieznosc jest OCZEKIWANA i znana: to remisy na medianie w filtrze")
print("    rownania (3), rozstrzygane inaczej przez pandas i przez add.reduceat)")
assert np.corrcoef(y_szybka[_ok], _y_ref[_ok])[0, 1] > 0.999, \
    "szybka implementacja rozjechala sie z zapisana krzywa bardziej niz 02-04.09"
print(f"\nWalidacja A: {time.time() - _t:.1f} s")

A1. Referencyjna z modulu (STARA) kontra finetune_mosc_...fullhistory.csv
   punktow: 200; z innym Np: 0; z innym Nm: 0; max |d log10 PPDF|: 2.887e-13

A2. Szybka z modulu (STARA) kontra ten sam plik, co dziesiaty punkt siatki
   punktow: 7433; korelacja 0.999977; |roznica| > 0,2 w 0 punktach; max 0.098
   (rozbieznosc jest OCZEKIWANA i znana: to remisy na medianie w filtrze
    rownania (3), rozstrzygane inaczej przez pandas i przez add.reduceat)

Walidacja A: 8.5 s


In [20]:
# KOMORKA 2 - WALIDACJA B: czy modul odtwarza konwencje ZESPOLU. Najmocniejsza
# kontrola, jaka mamy - porownanie z ZAPISANYM WYJSCIEM ich kodu, a nie z
# liczba przepisana z artykulu. Plik pdf_Mosc_2013-11-14 07-00-00.csv z
# archiwum CREDO zawiera 332 kosze z kolumnami cr mean / eq sum / cr delta3 /
# A / B / C dla opublikowanego punktu sekcji 3 Moskwy.
mosc_arch = load_cr(MOSC_ARCH_PATH)
eq_arch = load_eq(EQ_ARCH_PATH)

zespol = pd.read_csv(TABELA_ZESPOLU)
zespol["cr date"] = pd.to_datetime(zespol["cr date"])
r = st.kosze_ref(mosc_arch, eq_arch, T0_SEKCJA3, P3_DAYS, D_DAYS, DT_DAYS, konw=st.CREDO)
_m = r["wazne"]
tab = pd.DataFrame({"cr date": r["kraw_cr"][_m], "cr mean": r["cr_mean"][_m],
                    "eq sum": r["Sm"][_m], "cr delta3": r["dCR"][_m],
                    "A": r["A"][_m], "B": r["B"][_m], "C": r["C"][_m]})
_s = tab.merge(zespol, on="cr date", suffixes=("_my", "_zesp"), how="outer", indicator=True)
_ob = _s[_s["_merge"] == "both"]
print(f"wierszy: my {len(tab)}, zespol {len(zespol)}, wspolnych dat {len(_ob)}")
print(f"Np/Nm:   my {r['Np']}/{r['Nm']}, zespol "
      f"{int((zespol['C'] > 0).sum())}/{int((zespol['C'] < 0).sum())}")
print(f"PPDF:    my {r['PPDF']:.15e}  |  zespol 3.509338616358831e-08")
print(f"kotwice: CR {r['start_cr']} (jitter {r['jitter_cr_h']:+.3f} h), "
      f"EQ {r['start_eq']} (jitter {r['jitter_eq_h']:+.4f} h)")
print(f"EFEKTYWNE dt = {r['dt_efektywne_d']:.6f} dnia zamiast nominalnych {DT_DAYS}")
for kol in ["cr mean", "eq sum", "cr delta3", "A", "B", "C"]:
    print(f"   max |roznica| w kolumnie {kol:<10}: "
          f"{(_ob[f'{kol}_my'] - _ob[f'{kol}_zesp']).abs().max():.3e}")
assert len(tab) == len(zespol) == len(_ob), "kosze nie pokrywaja sie z tabela zespolu"
assert (_ob["C_my"] == _ob["C_zesp"]).all(), "znaki C nie zgadzaja sie kosz w kosz"
assert (r["Np"], r["Nm"]) == (214, 118), f"dostalismy {r['Np']}/{r['Nm']}, oczekiwano 214/118"
assert abs(r["PPDF"] / 3.509338616358831e-08 - 1) < 1e-9
print("\nOK - modul w konwencji CREDO odtwarza wyjscie kodu zespolu kosz w kosz.")

wierszy: my 332, zespol 332, wspolnych dat 332
Np/Nm:   my 214/118, zespol 214/118
PPDF:    my 3.509338616358796e-08  |  zespol 3.509338616358831e-08
kotwice: CR 2013-11-09 12:00:00 (jitter +5.000 h), EQ 2013-11-24 07:04:21 (jitter +0.0725 h)
EFEKTYWNE dt = 14.794688 dnia zamiast nominalnych 15
   max |roznica| w kolumnie cr mean   : 5.684e-14
   max |roznica| w kolumnie eq sum    : 1.137e-13
   max |roznica| w kolumnie cr delta3 : 1.776e-15
   max |roznica| w kolumnie A         : 2.220e-16
   max |roznica| w kolumnie B         : 1.776e-15
   max |roznica| w kolumnie C         : 0.000e+00

OK - modul w konwencji CREDO odtwarza wyjscie kodu zespolu kosz w kosz.


In [21]:
# KOMORKA 3 - WALIDACJA C: czy w konwencji CREDO szybka i referencyjna sa
# wymienne. W starej konwencji nie byly (remisy na medianie w filtrze rownania
# (3)). W CREDO tego filtra nie ma, wiec efekt powinien byc mniejszy - ale nie
# musi byc zerowy, bo remis A*B = 0 wciaz moze przesunac Np o jeden. Mierzymy
# to, zamiast zakladac.
_t = time.time()
_pk = np.sort(rng.choice(len(SIATKA), size=300, replace=False))
_wier = []
for i in _pk:
    rr = st.stat_ref(mosc_nasz, eq, SIATKA[i], P_DAYS, D_DAYS, M_THRESHOLD, DT_DAYS,
                     konw=st.CREDO)
    sz = st.punkt(cr_prep_mosc, eq_prep, SIATKA[i], OKNO_CREDO)
    _wier.append(dict(t0=SIATKA[i], Np_ref=rr["Np"], Np_sz=sz["Np"],
                      N_ref=rr["N_valid"], N_sz=sz["N_valid"],
                      remis_ref=rr["remisow"], remis_sz=sz["remisow"],
                      d_log10=abs(-np.log10(rr["PPDF"]) + np.log10(sz["PPDF"])),
                      dt_ref=rr["dt_efektywne_d"], dt_sz=sz["dt_efektywne_d"]))
walid = pd.DataFrame(_wier)
print(f"punktow: {len(walid)} ({time.time() - _t:.1f} s)")
print(f"  z innym Np:      {int((walid['Np_ref'] != walid['Np_sz']).sum())}")
print(f"  z innym N:       {int((walid['N_ref'] != walid['N_sz']).sum())}")
print(f"  max |d log10|:   {walid['d_log10'].max():.3e}")
print(f"  remisow (ref/sz): {int(walid['remis_ref'].sum())} / {int(walid['remis_sz'].sum())}")
print(f"  max |d dt_efektywne|: {(walid['dt_ref'] - walid['dt_sz']).abs().max():.2e} dnia")
walid.to_csv(f"{RESULTS_DIR}/credo_walidacja.csv", index=False)
assert (walid["dt_ref"] - walid["dt_sz"]).abs().max() < 1e-9, \
    "obie implementacje kotwicza krate inaczej - to bylby blad, nie szum"
assert walid["d_log10"].max() < 0.5, \
    "rozbieznosc powyzej 0,5 dekady - przyczyna inna niz remisy, zbadac osobno"
print("\nWniosek do zapisania w dzienniku: w konwencji CREDO obie implementacje")
print("zgadzaja sie co do kotwicy krat, a rozbieznosc PPDF pochodzi wylacznie")
print("z remisow. Skala jest wypisana wyzej - i to jest liczba, ktora wolno")
print("cytowac, a nie zalozenie, ze 'teraz juz sie zgadzaja'.")

punktow: 300 (10.9 s)
  z innym Np:      7
  z innym N:       0
  max |d log10|:   9.749e-02
  remisow (ref/sz): 136 / 122
  max |d dt_efektywne|: 0.00e+00 dnia

Wniosek do zapisania w dzienniku: w konwencji CREDO obie implementacje
zgadzaja sie co do kotwicy krat, a rozbieznosc PPDF pochodzi wylacznie
z remisow. Skala jest wypisana wyzej - i to jest liczba, ktora wolno
cytowac, a nie zalozenie, ze 'teraz juz sie zgadzaja'.


In [22]:
# KOMORKA 4 - NAJDROZSZA (~2-4 min na 20 rdzeniach). DWADZIESCIA KRZYWYCH
# PELNEJ HISTORII W KONWENCJI ARTYKULU. Jedno zadanie = jedna stacja; dane CR
# wczytywane wewnatrz zadania, zeby nie trzymac dwudziestu szeregow naraz.
#
# Siatka t0 jest DOKLADNIE ta sama, co w plikach z 26.08 - dzieki temu stara i
# nowa krzywa opisuja te same punkty i wolno je odjac. Pliki wyjsciowe maja
# przedrostek credo_, wiec nic ze starych nie jest nadpisywane, I MAJA TE SAME
# KOLUMNY co finetune_*.csv (N, N_valid, Np, Nm, PPDF, PCDF, sigma, t0) plus
# neglog10_PPDF i dt_efektywne_d. Ta zgodnosc jest celowa: dzieki niej porty
# notebookow b-i sprowadzaja sie do podmiany nazwy pliku, bez ruszania kodu
# analitycznego.
#
# Zapisujemy takze dt_efektywne. Przy kotwicy na danych krata przesuwa sie tam,
# gdzie w szeregu CR jest dziura - dla Moskwy w 1968 przesunela sie o 54 dni
# (dziennik 20260905.txt pkt 13f). Kolumna dt_efektywne_d pokazuje to wprost i
# jest diagnostyka jakosci danych stacji, a nie tylko szczegolem technicznym.
PRZELICZ_OD_NOWA = False   # True wymusza policzenie mimo istniejacych plikow


def podsumuj(stacja, y, N, dt_d, pominieta):
    """Diagnostyka liczona TYLKO na punktach, w ktorych okno faktycznie miesci
    sie w danych stacji (N_valid >= 90% koszy). Bez tego progu na krawedziach
    siatki - tam, gdzie stacja jeszcze nie istniala - kotwica skacze o tysiace
    dni i zalewa statystyke; to nie jest dziura w danych, tylko brak danych."""
    liczone = N > 0
    odrzucone = np.isfinite(dt_d) & ~liczone      # okno dalo sie ulozyc, ale prog je odrzucil
    if not liczone.any():
        return dict(stacja=stacja, pominieta=pominieta, punktow=len(N), liczonych=0)
    odch = np.abs(dt_d[liczone] - DT_DAYS)
    return dict(stacja=stacja, pominieta=pominieta, punktow=len(N),
                liczonych=int(liczone.sum()),
                odrzuconych_progiem=int(odrzucone.sum()),
                frakcja_odrzuconych=float(odrzucone.sum() / max(len(N), 1)),
                mediana_N_valid=float(np.median(N[liczone])),
                mediana_neglog10=float(np.nanmedian(y[liczone])),
                max_neglog10=float(np.nanmax(y[liczone])),
                dt_odchylenie_max=float(np.nanmax(odch)))


def zadanie_stacja(stacja):
    sciezka = f"{RESULTS_DIR}/credo_{stacja}_P3350_d5_dt15_fullhistory.csv"
    if not PRZELICZ_OD_NOWA and os.path.exists(sciezka):
        # Stacja policzona wczesniej (albo w przerwanym przebiegu) - diagnostyke
        # odtwarzamy z pliku, zeby wznowienie nie gubilo wierszy tabeli.
        d = pd.read_csv(sciezka)
        assert len(d) == len(siatka_stacji(stacja)), (
            f"{stacja}: zapisany plik ma {len(d)} wierszy, a siatka tej stacji "
            f"{len(siatka_stacji(stacja))} - to plik z poprzedniej, blednej wersji; "
            "usun pliki credo_*_fullhistory.csv i policz od nowa")
        return podsumuj(stacja, d["neglog10_PPDF"].to_numpy(),
                        d["N_valid"].to_numpy(), d["dt_efektywne_d"].to_numpy(), True)
    cr = load_station(stacja)
    siatka = siatka_stacji(stacja)
    siatka_ns = siatka.values.astype("datetime64[ns]").astype(np.int64)
    y, Np, Nm, N, dt_d = st.krzywa(st.przygotuj_szereg(cr), eq_prep, siatka_ns,
                                   OKNO_CREDO, max_jitter_d=MAX_JITTER_D)
    st.ramka_krzywej(siatka, y, Np, Nm, N, dt_d, OKNO_CREDO.n_koszy).to_csv(sciezka, index=False)
    return podsumuj(stacja, y, N, dt_d, False)


_t = time.time()
kotwice = run_jobs_parallel(zadanie_stacja, STACJE, opis="stacji", raport_co=2)
print(f"\n20 krzywych w konwencji CREDO: {(time.time() - _t) / 60:.2f} min")
kotwice = kotwice.sort_values("stacja").reset_index(drop=True)
kotwice.to_csv(f"{RESULTS_DIR}/credo_kotwice.csv", index=False)
print(kotwice.to_string(index=False, float_format=lambda x: f"{x:9.3f}"))
print(f"\nStacji policzonych w tym przebiegu: {int((~kotwice['pominieta']).sum())}, wczytanych z dysku: {int(kotwice['pominieta'].sum())}")
print("\nJAK CZYTAC. odrzuconych_progiem to punkty, w ktorych okno DALO SIE ulozyc,")
print("ale kotwica przeniosla je przez dziure w szeregu CR dalej niz o jeden kosz -")
print("liczylibysmy wtedy inne okno niz to, o ktore pytamy. Stacja z duza frakcja")
print("ma poszarpany szereg CR i jej krzywa jest lokalnie nieciagla. Kolumna")
print("dt_odchylenie_max dotyczy juz TYLKO punktow policzonych, wiec z definicji")
print(f"nie przekracza progu {MAX_JITTER_D} d.")

  2/20 stacji; minelo 0.1 min, zostalo ~1.1 min
  4/20 stacji; minelo 0.2 min, zostalo ~0.6 min
  6/20 stacji; minelo 0.2 min, zostalo ~0.4 min
  8/20 stacji; minelo 0.2 min, zostalo ~0.3 min
  10/20 stacji; minelo 0.3 min, zostalo ~0.3 min
  12/20 stacji; minelo 0.3 min, zostalo ~0.2 min
  14/20 stacji; minelo 0.4 min, zostalo ~0.2 min
  16/20 stacji; minelo 0.4 min, zostalo ~0.1 min
  18/20 stacji; minelo 0.4 min, zostalo ~0.0 min
  20/20 stacji; minelo 0.5 min, zostalo ~0.0 min

20 krzywych w konwencji CREDO: 0.47 min
stacja  pominieta  punktow  liczonych  odrzuconych_progiem  frakcja_odrzuconych  mediana_N_valid  mediana_neglog10  max_neglog10  dt_odchylenie_max
  aatb      False    62636      59443                 3193                0.051          656.000             2.144        13.576              5.000
  apty      False    22464      22464                    0                0.000          670.000             2.601        12.366              4.936
  athn      False    21933   

In [23]:
# KOMORKA 5 - KRZYWA MOSKWY NA PLIKU CR Z ARCHIWUM CREDO. To jest ta jedna
# krzywa, ktorej wolno uzywac do zestawien z opublikowanymi liczbami (decyzja
# Macka z 05.09: wersja pliku CR idzie tylko tam, gdzie porownujemy sie z
# artykulem). Plik z archiwum konczy sie 2019-12-11, wiec siatke przycinamy do
# t0, dla ktorych cale okno P + dt jeszcze sie w nim miesci - inaczej ostatnie
# kosze byly by puste i krzywa bylaby cicho zaniżona.
_kon = mosc_arch.index.max() - pd.Timedelta(days=P_DAYS)
_maska = SIATKA <= _kon
print(f"Plik z archiwum konczy sie {mosc_arch.index.max()}; ostatni pelny t0: {_kon}")
print(f"Siatka przycieta: {int(_maska.sum())} z {len(SIATKA)} punktow "
      f"({100 * _maska.mean():.1f}%), {SIATKA[_maska][0]} .. {SIATKA[_maska][-1]}")

_t = time.time()
y_arch, Np_arch, Nm_arch, N_arch, dt_arch = st.krzywa(
    st.przygotuj_szereg(mosc_arch), eq_prep, SIATKA_NS[_maska], OKNO_CREDO,
    max_jitter_d=MAX_JITTER_D)
krzywa_arch = st.ramka_krzywej(SIATKA[_maska], y_arch, Np_arch, Nm_arch, N_arch,
                               dt_arch, OKNO_CREDO.n_koszy)
krzywa_arch.to_csv(f"{RESULTS_DIR}/credo_mosc_archiwum_P3350_d5_dt15_fullhistory.csv",
                   index=False)
print(f"Policzona w {(time.time() - _t) / 60:.2f} min.")

# Kontrola: piec opublikowanych punktow sekcji 4 musi lezec na tej siatce i dac
# glebokosci zmierzone 05.09 (20260905c). Punkty kraty artykulu maja faze
# 07:37:12, a nasza siatka 6 h ich nie zawiera - wiec liczymy je wprost.
ARTYKUL = pd.DataFrame([
    dict(nr=1, t0=pd.Timestamp("1968-07-12 07:37:12"), PPDF=3.34e-8),
    dict(nr=2, t0=pd.Timestamp("1978-07-11 07:37:12"), PPDF=3.00e-6),
    dict(nr=3, t0=pd.Timestamp("1988-09-21 07:37:12"), PPDF=3.58e-5),
    dict(nr=4, t0=pd.Timestamp("1999-12-23 07:37:12"), PPDF=2.92e-6),
    dict(nr=5, t0=pd.Timestamp("2009-02-23 07:37:12"), PPDF=8.10e-9)])
_w = []
for _, a in ARTYKUL.iterrows():
    p = st.punkt(st.przygotuj_szereg(mosc_arch), eq_prep, a["t0"], OKNO_CREDO)
    _w.append(dict(nr=int(a["nr"]), t0=a["t0"], N=p["N_valid"], Np=p["Np"], Nm=p["Nm"],
                   nadwyzka=abs(p["Np"] - p["N_valid"] / 2),
                   znak="ordering" if p["Np"] > p["N_valid"] / 2 else "disordering",
                   neglog10=-np.log10(p["PPDF"]),
                   artykul=float(-np.log10(a["PPDF"])),
                   dt_efekt_d=p["dt_efektywne_d"]))
punkty_art = pd.DataFrame(_w)
punkty_art["roznica"] = punkty_art["artykul"] - punkty_art["neglog10"]
print("\nPiec opublikowanych punktow sekcji 4, CR z archiwum, konwencja artykulu:")
print(punkty_art.to_string(index=False, float_format=lambda x: f"{x:8.2f}"))
print(f"\nSuma |roznicy| po pieciu erach: {punkty_art['roznica'].abs().sum():.2f} dekady")
assert punkty_art["roznica"].abs().max() < 1.2, (
    "ktoras era rozjechala sie bardziej niz 05.09 (max bylo 0,77) - modul liczy "
    "co innego niz credo_kosze z 20260905c, trzeba to zbadac przed czytaniem reszty")

Plik z archiwum konczy sie 2019-12-11 06:00:00; ostatni pelny t0: 2010-10-09 06:00:00
Siatka przycieta: 66868 z 74322 punktow (90.0%), 1965-01-01 12:00:00 .. 2010-10-09 06:00:00
Policzona w 0.23 min.

Piec opublikowanych punktow sekcji 4, CR z archiwum, konwencja artykulu:
 nr                  t0   N  Np  Nm  nadwyzka        znak  neglog10  artykul  dt_efekt_d  roznica
  1 1968-07-12 07:37:12 670 266 404     69.00 disordering      7.72     7.48       13.75    -0.24
  2 1978-07-11 07:37:12 670 282 386     53.00 disordering      5.16     5.52       14.82     0.36
  3 1988-09-21 07:37:12 670 290 380     45.00 disordering      4.14     4.45       14.91     0.31
  4 1999-12-23 07:37:12 668 284 384     50.00 disordering      4.77     5.53       14.93     0.77
  5 2009-02-23 07:37:12 667 403 262     69.50    ordering      7.84     8.09       14.84     0.25

Suma |roznicy| po pieciu erach: 1.93 dekady


In [24]:
# KOMORKA 6 - CO ZMIENIA SAMA KONWENCJA. Obie krzywe kazdej stacji stoja na tej
# samej siatce t0 i na tych samych danych, wiec roznica miedzy nimi jest
# CZYSTYM efektem konwencji. Nic tu nie liczymy od nowa - tylko odejmujemy
# zapisane pliki.
_wier = []
for s_ in STACJE:
    stara = pd.read_csv(f"{RESULTS_DIR}/finetune_{s_}_P3350_d5_dt15_fullhistory.csv",
                        parse_dates=["t0"]).sort_values("t0").reset_index(drop=True)
    nowa = pd.read_csv(f"{RESULTS_DIR}/credo_{s_}_P3350_d5_dt15_fullhistory.csv",
                       parse_dates=["t0"]).sort_values("t0").reset_index(drop=True)
    # laczymy po t0, a nie po pozycji - siatki sa per stacja i nie ma powodu
    # zakladac, ze obie maja tyle samo wierszy
    z = stara[["t0", "PPDF"]].merge(nowa[["t0", "PPDF"]], on="t0",
                                    suffixes=("_st", "_no"), how="inner")
    y_st = -np.log10(z["PPDF_st"].to_numpy())
    y_no = -np.log10(z["PPDF_no"].to_numpy())
    ok = np.isfinite(y_st) & np.isfinite(y_no)
    d = y_no[ok] - y_st[ok]
    _wier.append(dict(stacja=s_, wspolnych_t0=len(z), punktow=int(ok.sum()),
                      korelacja=float(np.corrcoef(y_no[ok], y_st[ok])[0, 1]),
                      mediana_roznicy=float(np.median(d)),
                      p05=float(np.percentile(d, 5)), p95=float(np.percentile(d, 95)),
                      max_abs=float(np.max(np.abs(d))),
                      N_stara=float(np.median(stara["N_valid"])),
                      N_nowa=float(np.median(nowa["N_valid"]))))
porownanie = pd.DataFrame(_wier)
porownanie.to_csv(f"{RESULTS_DIR}/credo_vs_stara_krzywe.csv", index=False)
print("KRZYWA W KONWENCJI ARTYKULU KONTRA STARA, TE SAME DANE I TA SAMA SIATKA")
print("=" * 104)
print(porownanie.to_string(index=False, float_format=lambda x: f"{x:9.3f}"))
print("\nJAK TO CZYTAC. Korelacja blisko 1 oznacza, ze konwencja nie przestawia")
print("krzywej na inna - przesuwa ja. Mediana roznicy mowi, czy nowa konwencja")
print("systematycznie poglebia, czy splyca; rozstep p05-p95 - jak bardzo rusza")
print("pojedynczymi punktami. Kolumny N pokazuja, ze nowa konwencja liczy o")
print("jeden kosz wiecej (wiodacy kosz) i nie odrzuca remisow.")
print("\nUWAGA INTERPRETACYJNA: glebsza krzywa NIE znaczy lepsza. Sensowna miara")
print("jakosci jest tylko zgodnosc z opublikowanymi liczbami w opublikowanych")
print("punktach (komorka 5), a nie sam poziom krzywej.")

KRZYWA W KONWENCJI ARTYKULU KONTRA STARA, TE SAME DANE I TA SAMA SIATKA
stacja  wspolnych_t0  punktow  korelacja  mediana_roznicy       p05       p95   max_abs   N_stara    N_nowa
  aatb         62636    59443      0.984           -0.001    -0.521     0.556     4.791   653.000   655.000
  apty         22464    22464      0.986            0.010    -0.400     0.401     2.811   667.000   670.000
  athn         21933    21698      0.986           -0.014    -0.492     0.365     3.427   657.000   659.000
  fsmt         22084    22068      0.988            0.007    -0.372     0.345     3.976   667.000   670.000
  hrms         69752    69751      0.989           -0.003    -0.589     0.541     5.542   667.000   670.000
  invk         23192    23139      0.989            0.007    -0.387     0.449     3.234   663.000   666.000
  jung         74322    74310      0.983           -0.001    -0.416     0.402     3.546   667.000   670.000
 jung1         43644    43459      0.963            0.004    -0.

In [25]:
# KOMORKA 7 - PODSUMOWANIE DO DZIENNIKA.
print("=" * 84)
print("FUNDAMENT PRZELICZENIA W KONWENCJI ARTYKULU - PODSUMOWANIE")
print("=" * 84)
print("\n1. MODUL ../statystyka.py")
print(f"   STARA: {st.STARA}")
print(f"   CREDO: {st.CREDO}")
print(f"   okno P = {P_DAYS}: {OKNO_STARA.n_koszy} koszy w konwencji starej, "
      f"{OKNO_CREDO.n_koszy} w konwencji artykulu")
print("\n2. WALIDACJA")
print(f"   A1 referencyjna == cosmoseismic_stat na {len(_a1)} punktach: Np i Nm co do bitu")
print(f"   A2 szybka kontra zapisana krzywa: korelacja "
      f"{np.corrcoef(y_szybka[_ok], _y_ref[_ok])[0, 1]:.6f}, "
      f"{int(np.nansum(_d > 0.2))} punktow z |roznica| > 0,2")
print(f"   B  konwencja CREDO odtwarza tabele koszy zespolu: 332 kosze, 214/118, "
      f"PPDF do 15 cyfr")
print(f"   C  szybka kontra referencyjna w CREDO na {len(walid)} punktach: "
      f"{int((walid['Np_ref'] != walid['Np_sz']).sum())} punktow z innym Np, "
      f"max |d log10| {walid['d_log10'].max():.3e}")
print("\n3. KRZYWE ODNIESIENIA")
print(f"   {len(STACJE)} stacji, siatka {len(SIATKA)} punktow co 6 h, konwencja artykulu")
print(f"   -> ../results/credo_{{stacja}}_P3350_d5_dt15_fullhistory.csv")
print(f"   Moskwa na CR z archiwum, {int(_maska.sum())} punktow "
      f"-> credo_mosc_archiwum_P3350_d5_dt15_fullhistory.csv")
_naj = (kotwice.nlargest(3, "frakcja_odrzuconych")
        if "frakcja_odrzuconych" in kotwice else pd.DataFrame())
if len(_naj):
    print("   Stacje, ktorym prog higieny odrzucil najwiecej punktow (poszarpany CR):")
    for _, k in _naj.iterrows():
        print(f"     {k['stacja']}: {100 * k['frakcja_odrzuconych']:.1f}% siatki "
              f"({int(k['odrzuconych_progiem'])} punktow) - kotwica przenosila okno "
              f"dalej niz o kosz")
    print(f"   Lacznie odrzuconych progiem: {int(kotwice['odrzuconych_progiem'].sum())} "
          f"z {int(kotwice['punktow'].sum())} kandydatow t0 w calej sieci")
print("\n4. PIEC OPUBLIKOWANYCH PUNKTOW (CR z archiwum, konwencja artykulu)")
print(punkty_art[["nr", "t0", "N", "Np", "Nm", "znak", "nadwyzka",
                  "neglog10", "artykul", "roznica"]]
      .to_string(index=False, float_format=lambda x: f"{x:8.2f}"))
print(f"   suma |roznicy| = {punkty_art['roznica'].abs().sum():.2f} dekady na piec er")
print("\n5. CO DALEJ. Notebooki 20260906b-g czytaja pliki credo_*.csv jako krzywe")
print("   odniesienia. Kalibracja na 500 surogatach (20260902a) NIE jest")
print("   przeliczana - zostaje z adnotacja, ze policzona w starej konwencji.")

FUNDAMENT PRZELICZENIA W KONWENCJI ARTYKULU - PODSUMOWANIE

1. MODUL ../statystyka.py
   STARA: Konwencja(kotwica='nominalna', wiodacy_kosz=False, mediany='wszystkie', filtr='pelny', remisy='odrzucone')
   CREDO: Konwencja(kotwica='dane', wiodacy_kosz=True, mediany='po_filtrze', filtr='skrocony', remisy='w_N')
   okno P = 3350: 670 koszy w konwencji starej, 671 w konwencji artykulu

2. WALIDACJA
   A1 referencyjna == cosmoseismic_stat na 200 punktach: Np i Nm co do bitu
   A2 szybka kontra zapisana krzywa: korelacja 0.999977, 0 punktow z |roznica| > 0,2
   B  konwencja CREDO odtwarza tabele koszy zespolu: 332 kosze, 214/118, PPDF do 15 cyfr
   C  szybka kontra referencyjna w CREDO na 300 punktach: 7 punktow z innym Np, max |d log10| 9.749e-02

3. KRZYWE ODNIESIENIA
   20 stacji, siatka 74322 punktow co 6 h, konwencja artykulu
   -> ../results/credo_{stacja}_P3350_d5_dt15_fullhistory.csv
   Moskwa na CR z archiwum, 66868 punktow -> credo_mosc_archiwum_P3350_d5_dt15_fullhistory.csv
   St